# Assessing MESACLIP in the Arctic - Arctic Amplification

* **Description**: Reads in MESACLIP and Atmospheric Observational data for comparison of Arctic Amplification/temperature
* **Creator**: Alice DuVivier
* **Date**: December 2025

In [ ]:
import xarray as xr
import numpy as np
import pandas as pd
from glob import glob
from datetime import datetime
import xesmf as xe
import matplotlib.pyplot as plt

import dask
from distributed import Client
from ncar_jobqueue import NCARCluster

## Spin up DASK cluster

In [ ]:
# Create our NCAR Cluster - which uses PBSCluster under the hood
cluster = NCARCluster(walltime='2:00:00', cores=1, processes=1, memory='100 GB', 
                      resource_spec='select=1:ncpus=1:mem=100GB',interface='ext')

# Spin up 32 workers
cluster.scale(32)

# Assign the cluster to our Client
client = Client(cluster)

In [ ]:
client
# cluster.close()

## Set MESACLIP ensemble information

In [ ]:
# Historical
path_hist = '/gdex/data/d651007/'
# list 10 members of MESACLIP - last one is the first and named differently
ens_mems_hist = ['sehires38-1850-2005.001', 'hires1.0.30-1920-2005.002', 
                 'hires1.0.30-1920-2005.003', 'hires1.0.44-1920-2005.004', 
                 'hires1.0.44-1920-2005.005', 'hires1.0.45-1920-2005.006', 
                 'hires1.0.46-1920-2005.007', 'hires1.0.46-1920-2005.008',
                 'hires1.0.46-1920-2005.009', 'hires1.0.46-1920-2005.010']

## Load in ATMOSPHERIC Data
- T2m (TREFHT)

In [ ]:
%%time

print('loading historical data')
ds_hist = xr.Dataset()

for m in ens_mems_hist:
    print(m)
    # set case name
    case = 'b.e13.BHISTC5.ne120_t12.cesm-ihesp-' + m
    # set data directory
    path = path_hist+case+'/atm/proc/tseries/month_1'
    #print(path)
    # get list of files
    files = sorted(glob(f'{path}/{case}.cam.h0.TREFHT.??????-??????.nc'))       
    ds_tmp = xr.open_mfdataset(files, data_vars="minimal", coords='minimal', compat="override", parallel=True, 
                               concat_dim="time", combine='nested', decode_times=True)
    # take an average over the time bounds to get the right time dimension
    ds_tmp["time"] = ds_tmp.time_bnds.compute().mean(dim="nbnd")

    # keep only some variables
    keep_vars=['time_bound','time','lat','lon'] + ['TREFHT']
    ds_tmp = ds_tmp.drop_vars([v for v in ds_tmp.variables if v not in keep_vars])
    
    ds_hist = xr.concat([ds_hist,ds_tmp], dim='member_id')

In [ ]:
# concatenate datasets into single dataset
ds = xr.concat((ds_hist, ds_fut),dim='time')

# for whatever reason, the first ensemble is null, eliminate it
ds = ds.sel(member_id=slice(1,11))

In [ ]:
# save just atmospheric lat and lon arrays since they are time invariant
lat_atm = ds.lat.isel(time=100,member_id=0)
lon_atm = ds.lon.isel(time=100,member_id=0)

# keep just the variable of interest
ds = ds.TREFHT

In [ ]:
# crop the times to just 1920 and after
ds = ds.where(ds.time.dt.year >= 1920, drop=True)

In [ ]:
# Calculate the annual average
ds = ds.groupby('time.year').mean('time')

In [ ]:
# crop the latitudes to just >66N (the Arctic circle)
ds = ds.where(lat_atm >= 66.0)

In [ ]:
# now calculate spatial averages over the Arctic data
ds = ds.mean('ncol')

In [ ]:
# quick test plot
ds.isel(member_id=0).plot()
ds.isel(member_id=1).plot()

In [ ]:
# save this data for plotting later
ds_atm_model = ds

In [ ]:
ds_best.isel(year=0).plot()

In [ ]:
## Remapping functions from Kristen via Fred

In [ ]:
wgtfile = '/glade/work/fredc/Regrid_Maps/ne120np4_pentagons_to_latlon_0.25x0.25_0E_conserve_20200309.nc'
dsw = xr.open_dataset(wgtfile)

In [ ]:
def remap_camse(ds, dsw, varlst=[]):
    dso = xr.full_like(ds.drop_dims('ncol'), np.nan)
    lonb = dsw.xc_b.values.reshape([dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
    latb = dsw.yc_b.values.reshape([dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
    weights = sps.coo_matrix((dsw.S, (dsw.row-1, dsw.col-1)), shape=[dsw.sizes['n_b'], dsw.sizes['n_a']])
    if not varlst:
        for varname in list(ds):
            if 'ncol' in(ds[varname].dims):
                varlst.append(varname)
        if 'lon' in varlst: varlst.remove('lon')
        if 'lat' in varlst: varlst.remove('lat')
        if 'area' in varlst: varlst.remove('area')
    for varname in varlst:
        shape = ds[varname].shape
        invar_flat = ds[varname].values.reshape(-1, shape[-1])
        remapped_flat = weights.dot(invar_flat.T).T
        remapped = remapped_flat.reshape([*shape[0:-1], dsw.dst_grid_dims[1].values, dsw.dst_grid_dims[0].values])
        dimlst = list(ds[varname].dims[0:-1])
        dims={}
        coords={}
        for it in dimlst:
            dims[it] = dso.sizes[it]
            coords[it] = dso.coords[it]
        dims['lat'] = int(dsw.dst_grid_dims[1])
        dims['lon'] = int(dsw.dst_grid_dims[0])
        coords['lat'] = latb[:,0]
        coords['lon'] = lonb[0,:]
        remapped = xr.DataArray(remapped, coords=coords, dims=dims, attrs=ds[varname].attrs)
        dso = xr.merge([dso, remapped.to_dataset(name=varname)])
    varlst.clear() #not sure why this is needed but varlst was not cleared and it created issues with successive call to remap_camse
    return dso

In [ ]:
ds_ann_mean2 = remap_camse(ds_ann_mean,dsw)